In [ ]:
import pandas as pd
import seaborn as sns
import os
from sklearn.preprocessing import MinMaxScaler

attachments1_path = '附件1.xlsx'
attachments2_path = '附件2.xlsx'

In [ ]:
attachments1_sheet1 = pd.read_excel(attachments1_path, sheet_name=0)
attachments1_sheet2 = pd.read_excel(attachments1_path, sheet_name=1)
attachments2_sheet1 = pd.read_excel(attachments2_path, sheet_name=0)
attachments2_sheet2 = pd.read_excel(attachments2_path, sheet_name=1)

attachments2_sheet1=attachments2_sheet1.fillna(method='ffill')
merged_data_frame = pd.merge(attachments2_sheet1, attachments2_sheet2, on='作物编号', how='inner')
merged_data_frame = pd.merge(attachments2_sheet1, attachments2_sheet2, on='作物编号', suffixes=('_x', '_y'))


columns_to_keep = [col for col in merged_data_frame.columns if col.endswith('_x')]
columns_to_keep = [col.replace('_x', '') for col in columns_to_keep]

columns_to_drop = [col for col in merged_data_frame.columns if col.endswith('_y')]
merged_data_frame = merged_data_frame.drop(columns=columns_to_drop)
merged_data_frame.columns = [col.replace('_x', '') for col in merged_data_frame.columns]

In [ ]:
if '亩产量/斤' in merged_data_frame.columns and '销售单价/(元/斤)' in merged_data_frame.columns:
    merged_data_frame['销售量/斤'] = merged_data_frame['亩产量/斤'] * merged_data_frame['种植面积/亩']

merged_data_frame.to_excel('merged_file_with_sales.xlsx', index=False)
sales_merged_data_frame = merged_data_frame.groupby('作物名称').agg({'销售量/斤': 'sum'}).reset_index()
sales_merged_data_frame .to_excel('各农作物总销售量.xlsx', index=False)

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei'] 
plt.rcParams['axes.unicode_minus'] = False 

sales_merged_data_frame = pd.read_excel('merged_file_with_sales.xlsx')

In [ ]:
import numpy as np
def calculate_final_price(row):
    price_range = row['销售单价/(元/斤)'].split('-')
    min_price = float(price_range[0])
    max_price = float(price_range[1])
    average_price = (min_price + max_price) / 2
    random_fluctuation = np.random.uniform(-0.1, 0.1) * average_price
    final_price = average_price + random_fluctuation
    
    return final_price

data_frame =  merged_data_frame
data_frame['最终销售价格/(元/斤)'] = data_frame.apply(calculate_final_price, axis=1)
data_frame = data_frame.groupby('作物名称').agg({'最终销售价格/(元/斤)': 'mean','销售量/斤': 'sum'}).reset_index()
data_frame_2=pd.read_excel('初始数据未进行计算单价.xlsx')
data_frame_2['加权销售价格/(元/斤)'] = data_frame_2.apply(calculate_final_price, axis=1)

data_frame_2['总销量'] = data_frame_2['种植面积/亩'] * data_frame_2['亩产量/斤']
data_frame_2['总成本'] = data_frame_2['种植成本/(元/亩)'] * data_frame_2['种植面积/亩']
data_frame_2['总产量'] = data_frame_2['总销量']

In [ ]:
# 计算相关系数矩阵
corr = data_frame_2.select_dtypes(["float64", "int64"]).corr()

In [ ]:
Data= pd.read_excel('逻辑化数据.xlsx')
Data['最终销售价格/(元/斤)'] = Data.apply(calculate_final_price, axis=1)

In [ ]:
import numpy as np

data_frame=Data
data_frame['作物类型'] = pd.factorize(data_frame['作物类型'])[0]


ls1 = []  
ls2 = []  
ls3 = []  
ls4 = []  
ls5 = []  
ls6 = [] 
ls7 = [] 

def classify_row(row):
    if row['作物类型'] == 0 and row['作物名称'] != '水稻':
        ls1.append(row.name)
    elif row['作物名称'] == '水稻':
        ls2.append(row.name)
    elif (row['地块类型'] in [4, 5]) and row['作物类型'] != 2:
        ls6.append(row.name)
    elif row['作物类型'] == 2:
        ls5.append(row.name)
    elif row['种植季次'] == '第一季':
        ls3.append(row.name)
    elif row['种植季次'] == '第二季':
        ls4.append(row.name)
    if row['作物编号'] in [35, 36, 37]:
        ls7.append(row.name)

data_frame.apply(classify_row, axis=1)

bean = data_frame.index[data_frame['是否豆类'] == '是'].tolist()

In [ ]:
data_frame_1=pd.read_excel('附件1sheet1.xlsx')

In [ ]:
def find_indices(lst, element):
    return [index for index, value in enumerate(lst) if value == element]
def function(sales_min1,sales_min2,sales_min3):
    for i in sales_min.keys():
        total = 0
        index_ls = find_indices(land_types, i)

        total_product1 = sum([yields[j]*areas[i]*x[i][j][o]*0.25 for i in range(num_plots) for j in index_ls for o in range(2)])
        total_product2 = sum([yields[j]*areas[i]*x[i][j][o]*0.25 for i in range(num_plots) for j in index_ls for o in range(2,4)])
        total_product3 = sum([yields[j]*areas[i]*x[i][j][o]*0.25 for i in range(num_plots) for j in index_ls for o in range(4,6)])

        if total_product1 <= sales_min1[i]:
            pass
        else:
            total += (total_product-sales_min1[i])*prices[index_ls[0]]
        if total_product2 <= sales_min2[i]:
            pass
        else:
            total += (total_product-sales_min2[i])*prices[index_ls[0]]
            
        if total_product3 <= sales_min3[i]:
            pass
        else:
            total += (total_product-sales_min3[i])*prices[index_ls[0]]
    return total

In [ ]:
import pulp
model = pulp.LpProblem("Land_Type_Optimization", pulp.LpMaximize)

In [ ]:
num_plots = 54
num_crops = 125 

land_types =  data_frame['作物编号'].tolist() 
plot_type = data_frame_1['地块类型'].tolist() 
plot_area = data_frame_1['地块面积/亩'].tolist() 

In [ ]:
x = pulp.LpVariable.dicts(
    "planting_decision",                
    (range(num_plots), range(num_crops), range(6)),
    cat="Binary"                        
)
cost_per_acre = data_frame['种植成本/(元/亩)'].tolist()
cos = [cost_per_acre] * num_plots 

In [ ]:

growth_rates = {
    0: 0.003, 
    1: 0.0365,  
    2: 0.0719
}


sales_max = {}

for crop_id, group in data_frame.groupby('作物编号'):
    crop_type = group['作物类型'].iloc[0] 
    growth_rate = growth_rates.get(crop_type, 0)  

    current_sales = group['种植产量'].sum()
    max_sales = current_sales * (1 + growth_rate)

    sales_max[crop_id] = max_sales

In [ ]:
growth_rates = {
    0: 0.003,  
    1: 0.0365, 
    2: 0.0719 
}


random_range = (0, 0.03) 

sales_min = {}

for crop_id, group in data_frame.groupby('作物编号'):
    crop_type = group['作物类型'].iloc[0]  
    growth_rate = growth_rates.get(crop_type, 0) 


    current_sales = group['种植产量'].sum()
    max_sales = current_sales * (1 + growth_rate)
    

    min_growth_rate = -growth_rate
    random_adjustment = np.random.uniform(*random_range) 
    min_sales = current_sales * (1 + min_growth_rate) + random_adjustment*current_sales

    sales_min[crop_id] = min_sales


In [ ]:
sales = (data_frame.groupby('作物编号').sum()['种植产量']*1).to_dict()
costs=cos 
prices = (data_frame['亩产量/斤']*data_frame['平均销售单价']).tolist() 
yields = data_frame['亩产量/斤'].tolist() 
areas = data_frame_1['地块面积/亩'].tolist() 

In [ ]:
grain_crops = ls1 
rice_crop = ls2 
vegetable1_crops = ls3
vegetable2_crops = ls4 
fungi_crop = ls5
vegetable_crops = ls6

total_set = set(ls1 + ls2 + ls3 + ls4 + ls5 + ls6)

ex_grain_crops = list(total_set - set(grain_crops))
ex_rice_crop = list(total_set - set(rice_crop))
ex_vegetable1_crops = list(total_set - set(vegetable1_crops))
ex_vegetable2_crops = list(total_set - set(vegetable2_crops))
ex_fungi_crop = list(total_set - set(fungi_crop))
ex_vegetable_crops = list(total_set - set(vegetable_crops))
water_irrigated_second_season_crops = ls7 
water_irrigated_first_season_crops = list(set(vegetable1_crops) - set(vegetable2_crops))

bean_crops = bean
time = 6


In [ ]:
profit_sum = pulp.lpSum([
    (prices[j] * yields[j] - costs[i][j]) * areas[i] * x[i][j][t] 
    for i in range(num_plots) 
    for j in range(num_crops)
    for t in range(time) 
])

penalty = function(sales_min, sales_min, sales_min) 

model += profit_sum - penalty

In [ ]:
for i in range(num_plots): 
    if plot_type[i] in [0, 1, 2]: 
        for o in [0,2,4]:
            model += pulp.lpSum([x[i][j][o] for j in grain_crops]) ==1
            model += pulp.lpSum([x[i][j][o+1] for j in range(num_crops)]) == 0
        model += pulp.lpSum([x[i][j][o] for i in range(num_plots) for j in bean_crops for o in range(time)]) >= 1 
        
        for o in range(time):
            model += pulp.lpSum([x[i][j][o] for j in ex_grain_crops]) == 0

    elif plot_type[i] == 3:
  
        for o in [0,2,4]:
            model += pulp.lpSum([x[i][45][o]]) + pulp.lpSum([x[i][j][o] for j in vegetable1_crops]) == 1
            model += pulp.lpSum([x[i][45][o+1]]) == 0
            model += pulp.lpSum([x[i][j][o] for j in vegetable1_crops]) - pulp.lpSum([x[i][j][o+1] for j in vegetable2_crops]) <= 1 
            model += pulp.lpSum([x[i][45][o]]) + pulp.lpSum([x[i][j][o] for j in vegetable1_crops]) == 1 
            model += pulp.lpSum([x[i][45][o]]) + pulp.lpSum([x[i][j][o+1] for j in vegetable2_crops]) <= 1
            model += pulp.lpSum([x[i][45][o+1]]) == 0
            model += pulp.lpSum([x[i][j][o+1] for j in ex_vegetable2_crops]) == 0
            model += pulp.lpSum([x[i][j][o] for j in ex_vegetable1_crops]) == 0 
        model += pulp.lpSum([x[i][j][o] for i in range(num_plots) for j in bean_crops for o in range(time)]) >= 1 

    elif plot_type[i] == 4:
        for o in [0,2,4]:
            model += pulp.lpSum([x[i][j][o] for j in vegetable_crops]) == 1
            model += pulp.lpSum([x[i][j][o+1] for j in fungi_crop]) <=1  
            model += pulp.lpSum([x[i][j][o] for j in ex_vegetable_crops]) == 0 
            model += pulp.lpSum([x[i][j][o+1] for j in ex_fungi_crop]) == 0
        
        model += pulp.lpSum([x[i][j][0]+x[i][j][2]+x[i][j][4] for j in bean_crops]) >= 1

    elif plot_type[i] == 5:
        for o in [0,2,4]:
            model += pulp.lpSum([x[i][j][o] for j in vegetable1_crops]) == 1 
            model += pulp.lpSum([x[i][j][o+1] for j in vegetable1_crops]) == 1 
            model += pulp.lpSum([x[i][j][o] for j in ex_vegetable_crops]) == 0 
        

            
        model += pulp.lpSum([x[i][j][o] for j in bean_crops for o in range(time)]) == 1 

for j in range(num_crops):
    model += pulp.lpSum([x[i][j][o] for i in range(num_plots) for o in range(time)]) <= 2
    model += pulp.lpSum([x[i][j][o] for i in range(num_plots) if plot_type[i] == 4 for o in range(time)]) >= 2
model.solve()
output_folder = '问题1_1'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

a = data_frame_2[['作物名称', '作物类型']].drop_duplicates()
dic = {i: o for i, o in zip(data_frame_2.作物名称, data_frame_2.作物类型)}
reward = pd.DataFrame(index=data_frame['作物名称'].unique())
reward['作物类型'] = [dic[i] for i in reward.index]

start_year = 2024
end_year = 2030

with pd.ExcelWriter('问题1_1/result.xlsx', engine='openpyxl', mode='w') as writer:
    for wh in range(3):
        for t in range(6):
            current_year = 2023 + t//2 + 1 + wh*3
          
            if start_year <= current_year <= end_year:
                result = []
                product = []
                for i in range(num_plots):
                    tem = []
                    product_tem = []
                    for j in range(num_crops):
                        if pulp.value(x[i][j][t]) == 1:
                            tem.append(data_frame_1['地块面积/亩'].values[i])
                            product_tem.append(data_frame_1['地块面积/亩'].values[i] * data_frame['亩产量/斤'].values[j])
                        else:
                            tem.append(0)
                            product_tem.append(0)

                    result.append(tem)
                    product.append(product_tem)

                result_df = pd.DataFrame(result, index=data_frame_1['地块名称'].values, columns=data_frame['作物编号'].values).T
                result_df = result_df.groupby(result_df.index).sum().T
                result_df.columns = data_frame['作物名称'].unique()

                for i, plot_name in enumerate(result_df.index):
                    total_area = result_df.loc[plot_name].sum()  # 计算该地块所有作物的总种植面积
                    if total_area > plot_area[i]:  # 如果总面积超过限制
                        scale_factor = plot_area[i] / total_area  # 计算缩放因子
                        result_df.loc[plot_name] = result_df.loc[plot_name] * scale_factor  # 进行缩放调整

                sheet_name = f'{current_year}年{t%2 + 1}季'
                result_df.to_excel(writer, sheet_name=sheet_name)

                # 计算产量
                product_df = pd.DataFrame(product, index=data_frame_1['地块名称'].values, columns=data_frame['作物编号'].values).T
                product_df = product_df.groupby(product_df.index).sum().T
                product_df.columns = data_frame['作物名称'].unique()
                reward[f'{current_year}年{t%2 + 1}季产量（斤）'] = product_df.sum()

with pd.ExcelWriter('问题1_1/result.xlsx', engine='openpyxl', mode='a') as writer:
    reward.to_excel(writer, sheet_name='总产量')